# Shapes and node masks

Test points against a shape, select the nodes it covers, and visualize the result.

The `simgeometry.objects` package includes `Line` in 1D; `Circle`, `Rectangle`, `Triangle`, `Polygon`, and apertures in 2D; and `Sphere`, `Cuboid`, `Cylinder`, `Tube`, `Cone`, `Torus`, and apertures in 3D. The same `contains()`, `contains_points()`, and `node_mask()` pattern works across these shapes.

In [ ]:
import numpy as np

from simgeometry.grid import Grid
from simgeometry.objects import Circle

grid = Grid(2, (0.0, 0.0), (1.0, 1.0), (41, 41))
circle = Circle(midpoint=(0.5, 0.5), radius=0.25)
mask = circle.node_mask(grid)
indices = circle.get_indices_in_object(grid)
print("mask shape:", mask.shape)
print("selected nodes:", len(indices))
assert indices == grid.indices_from_mask(mask)

`node_mask()` is a host NumPy boolean array with the grid's shape. Use `get_indices_in_object()` when you need integer node indices instead.

In [ ]:
points = np.array([[0.5, 0.0, 0.8], [0.5, 0.0, 0.5]])
print("single point:", circle.contains((0.5, 0.5)))
print("point batch:", circle.contains_points(points))
assert circle.contains_points(points).tolist() == [True, False, False]

Batch points are component-major: the input shape is `(ndim, N)`, with one point per column. To see where the mask lies on the grid, plot its selected nodes.

In [ ]:
import matplotlib.pyplot as plt

nodes = grid.node_points()
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(nodes[0], nodes[1], c=mask.ravel(), cmap="coolwarm", s=9)
ax.set(xlabel="x", ylabel="y", title="Grid nodes selected by Circle.node_mask")
ax.set_aspect("equal")
plt.show()

## Inverted regions

Pass `inverted=True` to select the space outside a shape. The shape's bounding box still describes its original geometry.

In [ ]:
outside = Circle(midpoint=(0.5, 0.5), radius=0.25, inverted=True)
print("center selected:", outside.contains((0.5, 0.5)))
print("corner selected:", outside.contains((0.0, 0.0)))
assert np.array_equal(outside.node_mask(grid), ~mask)

## Solids in three dimensions

Besides `Sphere` and `Cuboid`, 3D geometry has solids of revolution along any axis: `Cylinder(start, end, radius)`, `Tube(start, end, inner_radius, outer_radius)` for rings and drift tubes, `Cone(start, end, start_radius, end_radius)` for cones and frustums, and `Torus(midpoint, major_radius, minor_radius, axis=...)`. `Cylinder` and `Tube` take an optional `fillet` that rounds their circular edges.

With `simgeometry[interactive]`, `show_3d(backend="pyvista")` draws a collection with one colour per object, and `plasma_plots.scene` shows it as an interactive scene. Click **Interactive 3-D view** below the picture, then rotate it to look along the tilted cylinder and into the bore of the tube.

In [ ]:
import plasma_plots
import pyvista as pv

from simgeometry import Cone, Cylinder, GeometryCollection, Torus, Tube

pv.OFF_SCREEN = True  # render without a window, e.g. on a docs server

solids = GeometryCollection(
    {
        "cylinder": Cylinder((-0.4, 0.4, -0.5), (0.4, -0.4, 0.5), 0.3),
        "tube": Tube((2, 0, -0.5), (2, 0, 0.5), 0.35, 0.65, fillet=0.1),
        "cone": Cone((0, 2, -0.5), (0, 2, 0.6), 0.6, 0.0),
        "torus": Torus((2, 2, 0), 0.55, 0.18, axis=(1, 0, 1)),
    }
)
assert solids["tube"].contains((2.5, 0, 0))
assert not solids["tube"].contains((2, 0, 0))  # the bore is empty
plotter = solids.show_3d(
    backend="pyvista", display=False, resolution=56, show_bounds=False
)
plasma_plots.scene(
    plotter,
    alt="A tilted cylinder, a tube with rounded edges, a cone and a tilted torus, "
    "one colour each",
)

`node_mask()` works the same way on a 3D grid. The nodes inside a tilted torus are drawn as points over its translucent surface; rotate the view to see that they fill the tube of the torus and leave its hole empty.

In [ ]:
grid3 = Grid(3, (-1.0, -1.0, -1.0), (1.0, 1.0, 1.0), (21, 21, 21))
torus = Torus((0, 0, 0), 0.6, 0.25, axis=(1, 0, 1))
mask3 = torus.node_mask(grid3)
print("selected nodes:", mask3.sum(), "of", mask3.size)
plotter = torus.show_3d(
    backend="pyvista", display=False, resolution=48, opacity=0.3, show_bounds=False
)
plotter.add_points(
    grid3.node_points()[:, mask3.ravel()].T,
    color="black",
    point_size=6,
    render_points_as_spheres=True,
)
plasma_plots.scene(
    plotter, alt="The grid nodes inside a tilted torus, drawn inside its surface"
)